In [1]:
import numpy as np 
import pandas as pd

In [2]:
# Setup workspace and generate baseline dataset
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

data = {
    'household_id': range(1, n + 1),
    'income': np.random.lognormal(mean=10.5, sigma=0.6, size=n),
    'wealth': np.random.lognormal(mean=11.5, sigma=1.0, size=n),
    'debt': np.random.exponential(scale=15000, size=n),
    'age': np.random.randint(25, 70, size=n),
    'stock_market_participant': np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    'liquidity_constrained': np.random.choice([0, 1], size=n, p=[0.7, 0.3])
}

df = pd.DataFrame(data)
df.loc[df['income'] < 20000, 'debt'] = np.nan

In [6]:
print(df.head)

<bound method NDFrame.head of      household_id         income         wealth          debt  age  \
0               1   48924.251431  400054.247877   7841.107096   35   
1               2   33424.398935  248856.992441   1024.340709   52   
2               3   53562.963185  104781.275909   6434.549959   56   
3               4   90564.529961   51692.253945   1764.838573   29   
4               5   31555.651338  198436.278729  24772.285651   62   
..            ...            ...            ...           ...  ...   
995           996   30679.189101  287837.176054   6292.969417   62   
996           997  106789.182840   96132.116776  29999.357731   66   
997           998   53343.409550   40868.928084   2524.953995   62   
998           999   25778.410187   83862.433261   1295.412958   47   
999          1000   51202.808796   46868.325097   9940.375253   49   

     stock_market_participant  liquidity_constrained  
0                           1                      0  
1                  

In [7]:
df_dropped = df.dropna(subset=['debt'])

# drop all rows where 'debt' is missing

In [9]:
print("--- [Method A] Dropping Missing Rows ---")
print("Original Row Count:", len(df))
print("Remaining Row Count after dropna:", len(df_dropped))

---[Method A] Dropping Missing Rows---
Original Row Count: 1000
Remaining Row Count after dropna: 852


In [10]:
median_debt = df['debt'].median()
df['debt_imputed'] = df['debt'].fillna(median_debt)

# replace missing debt with the median of non-missing debt values 

In [11]:
print("\n--- [Method B] Imputing Missing Debt with Median ---")
print("Median debt used for imputation:", round(median_debt, 2))
print("Missing count in debt_imputed", df['debt_imputed'].isna().sum())


---[Method B] Imputing Missing Debt with Median---
Median debt used for imputation: 10621.86
Missing count in debt_imputed 0


In [12]:
df['log_income'] = np.log(df['income'])
df['log_wealth'] = np.log(df['wealth'])

# create log-transformed income and wealth variables 
# np.log() applies natural logarithm: ln(x)

In [13]:
print("---[Step 2] Log Transformation Summary ---")
df[['income', 'log_income', 'wealth', 'log_wealth']].head()

---[Step 2] Log Transformation Summary ---


,income,log_income,wealth,log_wealth
0,48924.251431,10.798028,400054.247877,12.899355
1,33424.398935,10.417041,248856.992441,12.424634
2,53562.963185,10.888613,104781.275909,11.559630
3,90564.529961,11.413818,51692.253945,10.853063
4,31555.651338,10.359508,198436.278729,12.198223


In [15]:
df['income_quantile'] = pd.qcut(
    df['income'], 
    q = 5, 
    labels = ['Q1_Lowest', 'Q2_Low', 'Q3_Middle', 'Q4_High', 'Q5_Highest']
)

# split households into 5 equal-sized income quantiles (Q1 = Lowest 20 percent, Q5 = Highest 20 percent)

In [16]:
print("--- [Step 3] Income Quantile Distribution")
print(df['income_quantile'].value_counts().sort_index())

--- [Step 3] Income Quantile Distribution
income_quantile
Q1_Lowest     200
Q2_Low        200
Q3_Middle     200
Q4_High       200
Q5_Highest    200
Name: count, dtype: int64


In [17]:
df['dti_ratio'] = df['debt_imputed'] / df['income']

# compute debt to income (dti) ratio using imputed debt 

In [19]:
df['high_debt_dummy'] = np.where(df['dti_ratio'] > 0.5, 1, 0)

# generate a binary indicator: 1 if DTI > 0.5 (high debt burden), 0 otherwise
# np.where(condition, value_if_true, value_if_false)

In [20]:
print("--- [Step 4] High Debt Burden Proportion ---")
print(df['high_debt_dummy'].value_counts(normalize = True))

--- [Step 4] High Debt Burden Proportion ---
high_debt_dummy
0    0.642
1    0.358
Name: proportion, dtype: float64


In [21]:
df.head()

,household_id,income,wealth,debt,age,stock_market_participant,liquidity_constrained,debt_imputed,log_income,log_wealth,income_quantile,dti_ratio,high_debt_dummy
0,1,48924.251431,400054.247877,7841.107096,35,1,0,7841.107096,10.798028,12.899355,Q4_High,0.160270,0
1,2,33424.398935,248856.992441,1024.340709,52,1,0,1024.340709,10.417041,12.424634,Q3_Middle,0.030646,0
2,3,53562.963185,104781.275909,6434.549959,56,1,0,6434.549959,10.888613,11.559630,Q4_High,0.120131,0
3,4,90564.529961,51692.253945,1764.838573,29,0,0,1764.838573,11.413818,10.853063,Q5_Highest,0.019487,0
4,5,31555.651338,198436.278729,24772.285651,62,0,1,24772.285651,10.359508,12.198223,Q3_Middle,0.785035,1
